In [15]:
import cv2
import json
import os
import numpy as np
from PIL import Image

# Define directories
images_dir = 'data/test/images'  # Directory for original images
masks_dir = 'data/test/masks'    # Directory for mask images
output_json = 'data/test/tests_annotations_coco_format.json'  # Output JSON file

# COCO format template
coco_data = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "glasses"}]
}
annotation_id = 1
image_id = 1  # Reset image_id for each image

print("Started")

for image_filename in os.listdir(images_dir):
    if image_filename.endswith(".png"):
        # Paths to image and mask
        image_path = os.path.join(images_dir, image_filename)
        mask_path = os.path.join(masks_dir, image_filename)

        # Load image and mask
        image = Image.open(image_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Find contours (polygons) of the mask
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Store information about the image
        image_info = {
            "id": image_id,
            "file_name": image_filename,
            "height": image.height,
            "width": image.width
        }
        coco_data["images"].append(image_info)

        # Process each contour
        if contours:
            for contour in contours:
                # Flatten contour points into a list and convert to COCO format
                segmentation = contour.flatten().tolist()
                
                # Calculate area and bounding box
                x, y, w, h = cv2.boundingRect(contour)
                area = cv2.contourArea(contour)
    
                annotation = {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": 1,
                    "segmentation": [segmentation],
                    "area": area,
                    "boxes": [x, y, w, h],
                    "iscrowd": 0
                }
                coco_data["annotations"].append(annotation)
                annotation_id += 1

# Save annotations in COCO format JSON
with open(output_json, 'w') as f:
    json.dump(coco_data, f)

print("COCO format annotations saved to", output_json)


Started
COCO format annotations saved to data/test/tests_annotations_coco_format.json
